In [ ]:
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import regex as re
from tqdm import tqdm
import pandas as pd
import recordlinkage

path = "..\\..\\data\\processed\\hotel_general_info.csv"

In [ ]:
data0 = pd.read_csv(path)

In [ ]:
data0.head()

In [ ]:
count_city = data0['region'].value_counts()
print(count_city)

In [ ]:
data0.shape

In [ ]:
data = data0.dropna(subset=[
    'hotel_id',
    'hotel_name',
    'hotel_address',
    'region'
])

In [ ]:
data.shape

In [ ]:
def cut_district(addr):
    if(type(addr) == float):
            return ''
    if(len(addr.split(',')) < 3):
        return ''
    return addr.split(',')[-3]

def cut_city(addr):
    if(type(addr) == float):
            return ''
    return addr.split(',')[0]

data['district'] = data['hotel_address'].apply(cut_district)
data_district = data['district'].unique()
print(data_district)
print(len(data_district))

data['address'] = data['hotel_address'].apply(cut_city)
data_city = data['address'].unique()
print(data_city)
print(len(data_city))


In [ ]:
#Xử lý dấu cách
data["district"] = data["district"].apply(lambda x: re.sub(r"\s+", " ", x.strip()))
data['address'] = data['address'].apply(lambda x: re.sub(r"\s+", " ", x.strip()))

In [ ]:
data_district = data['district'].unique()
print(data_district)
print(len(data_district))

In [ ]:
data_city = data['address'].unique()
print(data_city)
print(len(data_city))

In [ ]:
district_count = data['district'].value_counts()

In [ ]:
print(district_count)

In [ ]:
# =========================
# PHÁT HIỆN DỮ LIỆU TRÙNG LẶP BẰNG RECORDLINKAGE
# =========================

# Bước 1: Tạo Indexer với Blocking Strategy
indexer = recordlinkage.Index()
indexer.block(['region', 'district', 'address'])  # Chỉ so sánh các record cùng region, district, address
candidates = indexer.index(data)

print(f"Tổng số cặp sau blocking: {len(candidates)}")
print(f"(Nếu không blocking sẽ có {len(data) * (len(data) - 1) // 2} cặp)")

In [ ]:
# Bước 2: Tạo Comparator và so sánh các cột
compare_cl = recordlinkage.Compare()


compare_cl.string("hotel_name", "hotel_name", method="levenshtein", threshold=0.8, label="hotel_name")


compare_cl.string("hotel_address", "hotel_address", method="levenshtein", threshold=0.7, label="hotel_address")


compare_cl.exact("hotel_id", "hotel_id", label="hotel_id")


features = compare_cl.compute(candidates, data)
print(f"Số cặp được so sánh: {len(features)}")
print("\nCác điểm khớp (0=không khớp, 1=khớp):")
print(features.head())

In [ ]:
# Bước 3: Lọc các cặp record trùng lặp
# Có 3 cột: hotel_name, hotel_address, hotel_id
# Chỉ chọn cặp cả 3 cột đều khớp hoặc ít nhất 2 cột khớp


matches_all = features[features.sum(axis=1) == 3]
print(f"Số cặp trùng lặp (3/3 cột khớp): {len(matches_all)}")

matches_2plus = features[features.sum(axis=1) >= 2]
print(f"Số cặp trùng lặp (≥2 cột khớp): {len(matches_2plus)}")


matches = matches_2plus
print(f"\nChọn sử dụng: {len(matches)} cặp trùng lặp")

In [ ]:
# Bước 4: In chi tiết các cặp trùng lặp
print("=" * 100)
print("CHI TIẾT CÁC CẶP TRÙNG LẶP")
print("=" * 100)

for idx, (x, y) in enumerate(matches.index[:20], 1):  # Hiển thị 20 cặp đầu tiên
    score = features.loc[(x, y)].sum()
    print(f"\nCặp {idx} (Điểm: {int(score)}/3):")
    print(f"  Record 1 (index={x}):")
    print(f"    Hotel ID: {data.iloc[x]['hotel_id']}")
    print(f"    Hotel Name: {data.iloc[x]['hotel_name']}")
    print(f"    Address: {data.iloc[x]['hotel_address']}")
    print(f"    Region: {data.iloc[x]['region']}")
    print(f"\n  Record 2 (index={y}):")
    print(f"    Hotel ID: {data.iloc[y]['hotel_id']}")
    print(f"    Hotel Name: {data.iloc[y]['hotel_name']}")
    print(f"    Address: {data.iloc[y]['hotel_address']}")
    print(f"    Region: {data.iloc[y]['region']}")
    print("-" * 100)

print(f"\nTỔNG HỢP:")
print(f"Tổng số cặp trùng lặp tìm được: {len(matches)}")
if len(matches) > 0:
    print(f"(Chỉ hiển thị 20 cặp đầu tiên, còn {len(matches) - 20} cặp khác)")

In [ ]:
# Bước 5: Lưu kết quả vào file CSV
if len(matches) > 0:
    duplicate_records = []
    for record_id_1, record_id_2 in matches.index:
        duplicate_records.append({
            'record_id_1': record_id_1,
            'record_id_2': record_id_2,
            'hotel_id_1': data.iloc[record_id_1]['hotel_id'],
            'hotel_id_2': data.iloc[record_id_2]['hotel_id'],
            'hotel_name_1': data.iloc[record_id_1]['hotel_name'],
            'hotel_name_2': data.iloc[record_id_2]['hotel_name'],
            'hotel_address_1': data.iloc[record_id_1]['hotel_address'],
            'hotel_address_2': data.iloc[record_id_2]['hotel_address'],
            'region_1': data.iloc[record_id_1]['region'],
            'region_2': data.iloc[record_id_2]['region'],
            'similarity_score': features.loc[(record_id_1, record_id_2)].sum()
        })
    
    duplicate_df = pd.DataFrame(duplicate_records)
    duplicate_path = "..\\..\\data\\processed\\duplicate_hotels_recordlinkage.csv"
    duplicate_df.to_csv(duplicate_path, index=False)
    print(f" Đã lưu danh sách trùng lặp vào: {duplicate_path}")
    print(f"\n THỐNG KÊ:")
    print(f"  - Tổng số record: {len(data)}")
    print(f"  - Số record duy nhất: {len(data) - len(duplicate_records)}")
    print(f"  - Số cặp trùng lặp: {len(duplicate_records)}")
    print(f"  - Tỷ lệ trùng lặp: {len(duplicate_records) * 2 / len(data) * 100:.2f}%")
else:
    print(" Không tìm thấy record trùng lặp")

In [ ]:
# Bước 6: Xoá các record bị trùng lặp
print("=" * 100)
print("XOÁ CÁC RECORD BỊ TRÙNG LẶP")
print("=" * 100)

if len(matches) > 0:
    # Lấy danh sách tất cả các index bị trùng lặp (giữ lại index đầu tiên, xoá các cái khác)
    records_to_delete = set()
    
    for record_id_1, record_id_2 in matches.index:
        # Giữ lại record_id_1, xoá record_id_2 (vì record_id_2 > record_id_1 luôn)
        records_to_delete.add(record_id_2)
    
    print(f"\n Danh sách records sẽ bị xoá: {sorted(records_to_delete)[:20]}...")  # Hiển thị 20 đầu tiên
    print(f"Tổng số records cần xoá: {len(records_to_delete)}")
    
    # Tạo dataframe mới mà không có các record trùng lặp
    data_cleaned = data.drop(index=list(records_to_delete))
    
    print(f"\n Kết quả:")
    print(f"  - Số records trước: {len(data)}")
    print(f"  - Số records xoá: {len(records_to_delete)}")
    print(f"  - Số records sau: {len(data_cleaned)}")
    print(f"  - Số records bị xoá giảm: {len(records_to_delete)} ({len(records_to_delete)/len(data)*100:.2f}%)")
    

    data = data_cleaned.reset_index(drop=True)
    print(f"\n Dataframe 'data' đã được cập nhật với {len(data)} records (không có trùng lặp)")
else:
    print(" Không có record trùng lặp để xoá")

In [ ]:

print("=" * 100)
print("LƯU DỮ LIỆU ĐÃ XOÁ TRÙNG LẶP")
print("=" * 100)


data_processed = data.copy()


out_path = "..\\..\\data\\processed\\hotel_general_info_processed.csv"
data_processed.to_csv(out_path, index=False)
print(f"\n Đã lưu dữ liệu đã xoá trùng lặp vào: {out_path}")
print(f"   - Tổng records: {len(data_processed)}")


